In [ ]:
import os
import re
import time
from urllib.parse import urljoin
import json
import pandas as pd

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials


from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
CLIENT_ID = '6037db494e7e459f9743e36800fe7c7c'
CLIENT_SECRET = '9f2deeadc47f4487a8b1374c28ff354b'

sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=CLIENT_ID, 
        client_secret=CLIENT_SECRET
        )
    )

# Test Spotify API Connection
try:
    results = sp.search(q='hello world', type='track', limit=1)
    if results['tracks']['items']:
        formatted_json = json.dumps(results, indent=2, ensure_ascii=False)
        print(formatted_json)
        print("================================================")
        print("Successfully connected to Spotify API!")
        print(results['tracks']['items'][0]['artists'][0]['id']) # artist id
        artist_id = results['tracks']['items'][0]['artists'][0]['id']
        print(f"Find: {results['tracks']['items'][0]['artists'][0]['name']}'s '{results['tracks']['items'][0]['name']}' ")
    else:
        print("Successfully connected to Spotify API! Nothing founded.")
except Exception as e:
    print(f"Failed to connected: {e}...")

## Fetch all country names

In [ ]:
def fetch_spotify_countries():
    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    driver.get("https://charts.spotify.com/charts/overview/global")

    input("Afterlogin please press Enter...")
    time.sleep(3)

    items = driver.find_elements(By.CSS_SELECTOR, 'li[role="option"][data-key]')
    print(f"Found {len(items)} country/region options")

    html_list = [li.get_attribute("outerHTML") for li in items]
    driver.quit()

    country_list = []
    for html in html_list:
        soup = BeautifulSoup(html, "html.parser")
        li = soup.find("li")
        if not li:
            continue

        data_key = li.get("data-key", "")
        name = li.get_text(strip=True)

        if data_key.startswith("/charts/overview/") and name:
            code = data_key.split("/")[-1]
            country_list.append({
                "code": code,
                "name": name,
                "data_key": data_key
            })

    return pd.DataFrame(country_list).drop_duplicates()

country_df = fetch_spotify_countries()
print(country_df.head(20))
print(country_df.shape)
country_df.to_csv("spotify_countries.csv", index=False)

## Fetch all countries top weekely songs
1. use the country dictionary previous we got
2. web script weekely data and save into csv

In [ ]:

# =========================================================
# Basic settings
# =========================================================
SPOTIFY_EMAIL = "0524andrea2405@gmail.com"

COUNTRY_CSV_PATH = "spotify_countries.csv"   
OUTPUT_DIR = "spotify_weekly_top_songs"      
PROGRESS_FILE = "progress.json"                                

BASE_URL = "https://charts.spotify.com/home"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# Build driver
# =========================================================
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# =========================================================
# 1) Import COUNTRY_MAP from csv file
#    key = code, value = name
# =========================================================
def load_country_map(csv_path):
    df = pd.read_csv(csv_path)
    required_cols = {"code", "name"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"CSV must contain columns: {required_cols}")

    country_map = dict(zip(df["code"].astype(str).str.strip(), df["name"].astype(str).str.strip()))
    return country_map, df

COUNTRY_MAP, country_df = load_country_map(COUNTRY_CSV_PATH)
weekly_top_music_by_country = {code: None for code in COUNTRY_MAP.keys()}

# =========================================================
# Utilities
# =========================================================
def sanitize_filename(text):
    text = str(text).strip()
    text = re.sub(r"[\\/:\*\?\"<>\|]", "_", text)
    text = re.sub(r"\s+", "_", text)
    return text

def get_output_csv_path(country_code, country_name):
    safe_name = sanitize_filename(country_name)
    return os.path.join(OUTPUT_DIR, f"{country_code}_{safe_name}_weekly_top_songs.csv")

def load_progress(progress_file):
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "completed": [],
        "failed": [],
        "last_country": None
    }

def save_progress(progress_file, progress_data):
    with open(progress_file, "w", encoding="utf-8") as f:
        json.dump(progress_data, f, ensure_ascii=False, indent=2)

progress = load_progress(PROGRESS_FILE)


# =========================================================
# 2) Open home page
# =========================================================
def open_spotify_charts_home(driver):
    driver.get(BASE_URL)
    time.sleep(2)


# =========================================================
# 3) Login flow 
# =========================================================
def spotify_login(driver, wait, email):
    print("Entering login flow...")

    login_btn = wait.until(
        EC.presence_of_element_located((
            By.XPATH,
            "//span[normalize-space()='Log in']/ancestor::button[1] | "
            "//span[normalize-space()='Log in']/ancestor::a[1] | "
            "//button[contains(., 'Log in')] | //a[contains(., 'Log in')]"
        ))
    )

    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", login_btn)
    time.sleep(1)
    driver.execute_script("arguments[0].click();", login_btn)

    email_input = wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'input[data-testid="login-username"]'))
    )
    email_input.clear()
    email_input.send_keys(email)

    continue_btn = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[data-testid="login-button"]'))
    )
    continue_btn.click()

    print("Please complete the login / verification in the browser.")
    input("After completion, return to the notebook / terminal and press Enter to continue...")
    print("Login completed.")


# =========================================================
# 4) Click Weekly Top Songs
# =========================================================
def click_weekly_top_songs(driver, wait):
    possible_xpaths = [
        "//a[contains(., 'Weekly Top Songs')]",
        "//button[contains(., 'Weekly Top Songs')]",
        "//span[contains(., 'Weekly Top Songs')]/ancestor::*[self::a or self::button][1]",
        "//*[contains(text(), 'Weekly Top Songs')]"
    ]

    for xp in possible_xpaths:
        try:
            elem = wait.until(EC.element_to_be_clickable((By.XPATH, xp)))
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", elem)
            time.sleep(1)
            driver.execute_script("arguments[0].click();", elem)
            time.sleep(3)
            print("Weekly Top Songs clicked")
            return
        except Exception:
            continue

    raise Exception("Weekly Top Songs button not found")

# =========================================================
# 5) Choose country
# =========================================================
def clear_input_mac_or_windows(search_input):
    try:
        # Mac
        search_input.send_keys(Keys.COMMAND, "a")
        time.sleep(0.3)
        search_input.send_keys(Keys.DELETE)
    except Exception:
        # Windows fallback
        search_input.send_keys(Keys.CONTROL, "a")
        time.sleep(0.3)
        search_input.send_keys(Keys.DELETE)

def select_country(driver, wait, country_code, country_name):
    print(f"Choosing country: {country_code} - {country_name}")

    # Search box
    search_input = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, 'input[data-testid="search-input"]'))
    )

    search_input.click()
    time.sleep(0.5)
    clear_input_mac_or_windows(search_input)
    time.sleep(0.5)

    search_input.send_keys(country_name)
    time.sleep(2)

    option_locator = (
        By.XPATH,
        f"//div[@role='option' and contains(., '{country_name}')] | "
        f"//li[@role='option' and contains(., '{country_name}')] | "
        f"//*[contains(@role, 'option') and contains(., '{country_name}')]"
    )

    committed = False

    # Method 1: Try keyboard selection first
    try:
        wait.until(EC.presence_of_element_located(option_locator))
        time.sleep(0.5)
        search_input.send_keys(Keys.ARROW_DOWN)
        time.sleep(0.3)
        search_input.send_keys(Keys.ENTER)
        committed = True
    except Exception:
        pass

    # Method 2: Directly click option
    if not committed:
        option_xpaths = [
            f"//div[@role='option' and contains(., '{country_name}')]",
            f"//li[@role='option' and contains(., '{country_name}')]",
            f"//*[contains(@role, 'option') and contains(., '{country_name}')]",
        ]

        for xp in option_xpaths:
            try:
                option = wait.until(EC.element_to_be_clickable((By.XPATH, xp)))
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", option)
                time.sleep(0.5)
                driver.execute_script("arguments[0].click();", option)
                committed = True
                break
            except Exception:
                continue

    # Method 3: Directly press Enter
    if not committed:
        search_input.send_keys(Keys.ENTER)
        committed = True

    time.sleep(2)

    final_value = driver.find_element(
        By.CSS_SELECTOR,
        'input[data-testid="search-input"]'
    ).get_attribute("value")

    print(f"Country selected: {country_name} | input value = {final_value}")


# =========================================================
# 6) Scroll to bottom, to ensure table is fully rendered
# =========================================================
def scroll_to_bottom(driver, pause_time=2, max_rounds=10):
    last_height = driver.execute_script("return document.body.scrollHeight")

    for _ in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause_time)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height


# =========================================================
# 7) Scrape raw table data
#    Expected to capture 6 columns:
#    rank, track_artist, peak, prev, streak, streams
# =========================================================
def scrape_chart_table_raw(driver):
    rows_data = []

    scroll_to_bottom(driver, pause_time=2, max_rounds=8)
    time.sleep(2)

    possible_row_xpaths = [
        "//table//tr",
        "//div[@role='row']"
    ]

    rows = []
    for xp in possible_row_xpaths:
        found = driver.find_elements(By.XPATH, xp)
        if found:
            rows = found
            break

    for row in rows:
        try:
            cells = row.find_elements(By.XPATH, ".//td | .//div[@role='cell']")
            texts = [c.text.strip() for c in cells if c.text.strip()]

            if len(texts) >= 6:
                rows_data.append(texts[:6])

        except Exception:
            continue

    return rows_data


# =========================================================
# 8) Single Country Process
# =========================================================
# ORIGINAL VERSION
# def process_one_country(driver, wait, country_code, country_name):
#     select_country(driver, wait, country_code, country_name)
#     time.sleep(2)

#     raw_rows = scrape_chart_table_raw(driver)
#     print(f"{country_code} Raw data scraped: {len(raw_rows)} rows")
#     df_country = pd.DataFrame(raw_rows)  
#     
#     return df_country

def process_one_country(driver, wait, country_code, country_name):
    select_country(driver, wait, country_code, country_name)
    time.sleep(2)

    raw_rows = scrape_chart_table_raw(driver)
    print(f"{country_code} Raw data scraped: {len(raw_rows)} rows")

    expected_cols = ['rank', 'track_artist', 'peak', 'prev', 'streak', 'streams']
    df_country = pd.DataFrame(raw_rows, columns=expected_cols)

    df_country = df_country.apply(lambda col: col.astype(str).str.strip())

    df_country['rank'] = df_country['rank'].str.extract(r'(\d+)', expand=False)

    df_country[['track_name', 'artist_name']] = (
        df_country['track_artist']
        .astype(str)
        .str.split('\n', n=1, expand=True)
    )

    df_country['track_name'] = df_country['track_name'].str.strip()
    df_country['artist_name'] = df_country['artist_name'].str.strip()

    df_country['streams'] = (
        df_country['streams']
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.extract(r'(\d+)', expand=False)
    )

    for col in ['rank', 'peak', 'prev', 'streak', 'streams']:
        df_country[col] = pd.to_numeric(df_country[col], errors='coerce')

    return df_country


# =========================================================
# 9) Main flow with progress tracking and error handling
# =========================================================
all_country_frames = []

try:
    open_spotify_charts_home(driver)
    spotify_login(driver, wait, SPOTIFY_EMAIL)
    click_weekly_top_songs(driver, wait)

    completed_set = set(progress.get("completed", []))
    failed_set = set(progress.get("failed", []))

    for country_code, country_name in COUNTRY_MAP.items():

        if country_code in completed_set:
            print(f"[SKIP] {country_code} - {country_name} had already finished, skipping...")
            csv_path = get_output_csv_path(country_code, country_name)
            if os.path.exists(csv_path):
                weekly_top_music_by_country[country_code] = pd.read_csv(csv_path)
            continue

        try:
            print("=" * 80)
            print(f"Starting: {country_code} - {country_name}")

            df_country = process_one_country(driver, wait, country_code, country_name)
            weekly_top_music_by_country[country_code] = df_country

            # save individual country csv
            csv_path = get_output_csv_path(country_code, country_name)
            df_country.to_csv(csv_path, index=False, encoding="utf-8-sig")
            print(f"Saved: {csv_path}")

            # all_country_frames
            df_tmp = df_country.copy()
            df_tmp["country_code"] = country_code
            df_tmp["country_name"] = country_name
            all_country_frames.append(df_tmp)

            # Update progress
            completed_set.add(country_code)
            if country_code in failed_set:
                failed_set.remove(country_code)

            progress["completed"] = sorted(list(completed_set))
            progress["failed"] = sorted(list(failed_set))
            progress["last_country"] = country_code
            save_progress(PROGRESS_FILE, progress)

            print(f"[DONE] {country_code} - {country_name}")

        except Exception as e:
            print(f"[ERROR] {country_code} - {country_name}: {e}")

            failed_set.add(country_code)
            progress["completed"] = sorted(list(completed_set))
            progress["failed"] = sorted(list(failed_set))
            progress["last_country"] = country_code
            save_progress(PROGRESS_FILE, progress)

            continue


except KeyboardInterrupt:
    print("Program manually stopped, progress.json and individual country csv files are preserved.")

finally:
    print("Process completed.")

In [ ]:
def rename_columns_and_clean_data(df):
    df = df.rename(columns={
        '0': 'rank',
        '1': 'track_artist',
        '2': 'peak',
        '3': 'prev',
        '4': 'streak',
        '5': 'streams'
    })

    df[['track_name', 'artist_name']] = (
        df['track_artist']
        .astype(str)
        .str.split('\n', n=1, expand=True)
    )

    df['track_name'] = df['track_name'].str.strip()
    df['artist_name'] = df['artist_name'].str.strip()

    df['streams'] = (
        df['streams']
        .astype(str)
        .str.replace(',', '', regex=False)
    )

    for col in ['rank', 'peak', 'prev', 'streak', 'streams']:
        df[col] = pd.to_numeric(
            df[col],
            errors='coerce'
        )
    return df

for country_code, country_name in COUNTRY_MAP.items():
    if country_code in progress.get("completed", []):
        print(f"[CLEAN] {country_code} - {country_name}")
        csv_path = get_output_csv_path(country_code, country_name)
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            cleaned_df = rename_columns_and_clean_data(df)
            cleaned_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
            weekly_top_music_by_country[country_code] = cleaned_df
            print(f"Cleaned and saved: {csv_path}")

# Main Funcitons

## 0. Read every information from csv

In [ ]:
from pathlib import Path

base_dir = Path.cwd()  
csv_dir = base_dir / "spotify_weekly_top_songs"

weekly_top_music_by_country = {}

for csv_file in csv_dir.glob("*.csv"):
    filename = csv_file.stem
    key = filename.split("_")[0]
    df = pd.read_csv(csv_file)
    df["rank"] = df.index + 1
    weekly_top_music_by_country[key] = df

for k, v in weekly_top_music_by_country.items():
    print(f"{k}:")
    print(v.head())
    print("-" * 40)

In [ ]:
df_now = weekly_top_music_by_country['tw']
df_now['rank'] = df_now.index + 1
df_now
print(df_now.columns)

## 1. Get needed informations

Already have ['rank', 'track_artist', 'peak', 'prev', 'streak', 'streams', 'track_name', 'artist_name']

Need duration_ms, explicit, is_local, release_date, artist_id, track_id

### 1.1 try one df

In [ ]:
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

# =========================
# 0. Spotify client
# =========================
auth_manager = SpotifyClientCredentials(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

sp = spotipy.Spotify(
    auth_manager=auth_manager,
    requests_timeout=10,
    retries=3
)

# =========================
# 1. function: query one row
# =========================
def get_track_metadata(row, sp):
    track_name = str(row["track_name"]).strip()
    artist_name = str(row["artist_name"]).strip()

    default_result = pd.Series({
        "duration_ms": pd.NA,
        "explicit": pd.NA,
        "is_local": pd.NA,
        "release_date": pd.NA,
        "artist_id": pd.NA,
        "track_id": pd.NA
    })

    if not track_name or track_name.lower() == "nan":
        return default_result

    try:
        q = f'track:"{track_name}" artist:"{artist_name}"'
        result = sp.search(q=q, type="track", limit=1)

        items = result.get("tracks", {}).get("items", [])
        if not items:
            return default_result

        track = items[0]

        return pd.Series({
            "duration_ms": track.get("duration_ms"),
            "explicit": track.get("explicit"),
            "is_local": track.get("is_local"),
            "release_date": track.get("album", {}).get("release_date"),
            "artist_id": track.get("artists", [{}])[0].get("id"),
            "track_id": track.get("id")
        })

    except Exception as e:
        print(f"Error on track='{track_name}', artist='{artist_name}': {e}")
        return default_result

# =========================
# 2. apply back to df
# =========================
df_now[[
    "duration_ms",
    "explicit",
    "is_local",
    "release_date",
    "artist_id",
    "track_id"
]] = df_now.apply(get_track_metadata, axis=1, sp=sp)
df_now["duration_ms"] = pd.to_numeric(df_now["duration_ms"], errors="coerce")
df_now["duration_sec"] = (df_now["duration_ms"] / 1000).round().astype("Int64")

print(df_now.head())

### 1.2 try every df

In [ ]:
import os
import re
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

# =========================
# 0. Spotify client
# =========================
auth_manager = SpotifyClientCredentials(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

sp = spotipy.Spotify(
    auth_manager=auth_manager,
    requests_timeout=10,
    retries=3
)

# =========================
# 1. query one row
# =========================
def get_track_metadata(row, sp):
    track_name = str(row["track_name"]).strip()
    artist_name = str(row["artist_name"]).strip()

    default_result = pd.Series({
        "duration_ms": pd.NA,
        "explicit": pd.NA,
        "is_local": pd.NA,
        "release_date": pd.NA,
        "artist_id": pd.NA,
        "track_id": pd.NA
    })

    if not track_name or track_name.lower() == "nan":
        return default_result

    try:
        q = f'track:"{track_name}" artist:"{artist_name}"'
        result = sp.search(q=q, type="track", limit=1)

        time.sleep(0.2)

        items = result.get("tracks", {}).get("items", [])
        if not items:
            return default_result

        track = items[0]

        return pd.Series({
            "duration_ms": track.get("duration_ms"),
            "explicit": track.get("explicit"),
            "is_local": track.get("is_local"),
            "release_date": track.get("album", {}).get("release_date"),
            "artist_id": track.get("artists", [{}])[0].get("id"),
            "track_id": track.get("id")
        })

    except Exception as e:
        print(f"Error on track='{track_name}', artist='{artist_name}': {e}")
        return default_result


# =========================
# 2. clean filename
# =========================
def clean_filename(name):
    name = str(name).strip()
    name = re.sub(r'[\\/*?:"<>|]', "_", name)   
    name = re.sub(r"\s+", "_", name)            
    return name


# =========================
# 3. process one dataframe
# =========================
def enrich_one_df(df, sp):
    df = df.copy()

    # 補 metadata
    df[[
        "duration_ms",
        "explicit",
        "is_local",
        "release_date",
        "artist_id",
        "track_id"
    ]] = df.apply(get_track_metadata, axis=1, sp=sp)

    df["duration_ms"] = pd.to_numeric(df["duration_ms"], errors="coerce")
    df["duration_sec"] = (df["duration_ms"] / 1000).round().astype("Int64")

    text_cols = ["release_date", "artist_id", "track_id"]
    df[text_cols] = df[text_cols].fillna("unknown")

    bool_cols = ["explicit", "is_local"]
    df[bool_cols] = df[bool_cols].astype("object").fillna("unknown")

    return df


# =========================
# 4. process dictionary and save csv
# =========================
def enrich_dict_and_save_csv(df_dict, sp, output_folder="spotify_weekly_top_song_all_info"):
    os.makedirs(output_folder, exist_ok=True)

    enriched_dict = {}

    for key, df in df_dict.items():
        print(f"Processing: {key}")

        try:
            enriched_df = enrich_one_df(df, sp)

            enriched_dict[key] = enriched_df

            safe_name = clean_filename(key)
            output_path = os.path.join(output_folder, f"{safe_name}.csv")
            enriched_df.to_csv(output_path, index=False, encoding="utf-8-sig")

            print(f"Saved: {output_path}")

        except Exception as e:
            print(f"Failed on {key}: {e}")

    return enriched_dict


all_country_dfs_enriched = enrich_dict_and_save_csv(
    df_dict=weekly_top_music_by_country,
    sp=sp,
    output_folder="spotify_weekly_top_song_all_info"
)
all_country_dfs_enriched["taiwan"].head()